# Week 1 Day 5: Brochure using Ollama

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

In [ ]:
# imports

# use sys when getting ModuleNotFoundError: No module named 'scraper', 
# this notebook lives two directory deeper

import sys
sys.path.insert(0, '../..')

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display 
from scraper import fetch_website_links, fetch_website_contents 
from openai import OpenAI

In [3]:
## use an Open Source model running locally via Ollama rather than OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

MODEL = "gemma4"

In [4]:
# Use a website link that you can scrape
# Check the robots.txt: Append /robots.txt to the main domain of any of these sites (e.g., huggingface.co/robots.txt). 
# If the site explicitly disallows scrapers, you must honor it.

links = fetch_website_links("https://www.maket.ai/")
links

['./',
 'https://app.maket.ai/register',
 './',
 'https://app.maket.ai/register',
 'https://app.maket.ai/register',
 './faqs',
 'https://app.maket.ai/register',
 'https://app.maket.ai/register',
 './',
 './about',
 './blog',
 './brand-kit',
 './features',
 './pricing',
 './contact',
 './releases',
 './legal/terms-and-conditions',
 './legal/privacy-policy',
 'https://x.com/maketplans',
 'https://www.facebook.com/Maketplans',
 'https://www.instagram.com/maket.ai',
 'https://ca.pinterest.com/maketaiplans/',
 'https://www.linkedin.com/company/maket-ai',
 'mailto:hello@maket.ai',
 'https://app.maket.ai/register',
 './',
 './about',
 './blog',
 './brand-kit',
 './features',
 './pricing',
 './contact',
 './releases',
 './legal/terms-and-conditions',
 './legal/privacy-policy',
 'https://x.com/maketplans',
 'https://www.facebook.com/Maketplans',
 'https://www.instagram.com/maket.ai',
 'https://ca.pinterest.com/maketaiplans/',
 'https://www.linkedin.com/company/maket-ai',
 'mailto:hello@maket.ai

In [5]:
# FIRST step: Have gemma4 read the links on the webpage and respond in structured JSON
# It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about". 
# Use "one shot prompting" in which we provide an example of how it should respond in the prompt.

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/career"}
        {"type": "pricing page", "url": "https://another.full.url/pricing"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://www.maket.ai"))


Here is the list of links on the website https://www.maket.ai -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

./
https://app.maket.ai/register
./
https://app.maket.ai/register
https://app.maket.ai/register
./faqs
https://app.maket.ai/register
https://app.maket.ai/register
./
./about
./blog
./brand-kit
./features
./pricing
./contact
./releases
./legal/terms-and-conditions
./legal/privacy-policy
https://x.com/maketplans
https://www.facebook.com/Maketplans
https://www.instagram.com/maket.ai
https://ca.pinterest.com/maketaiplans/
https://www.linkedin.com/company/maket-ai
mailto:hello@maket.ai
https://app.maket.ai/register
./
./about
./blog
./brand-kit
./features
./pricing
./contact
./releases
./legal/terms-and-conditions
./legal/privacy-policy
https://x.com/maketplans
https://www.facebook.com/Maketplans
http

In [9]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [10]:
select_relevant_links("https://www.maket.ai")

{'links': [{'type': 'about page', 'url': './about'},
  {'type': 'blog', 'url': './blog'},
  {'type': 'brand kit', 'url': './brand-kit'},
  {'type': 'features page', 'url': './features'},
  {'type': 'pricing page', 'url': './pricing'},
  {'type': 'contact us page', 'url': './contact'},
  {'type': 'social media link', 'url': 'https://x.com/maketplans'},
  {'type': 'social media link', 'url': 'https://www.facebook.com/Maketplans'},
  {'type': 'social media link', 'url': 'https://www.instagram.com/maket.ai'},
  {'type': 'social media link',
   'url': 'https://ca.pinterest.com/maketaiplans/'},
  {'type': 'company profile page',
   'url': 'https://www.linkedin.com/company/maket-ai'}]}

In [67]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://www.maket.ai")

In [ ]:
select_relevant_links("https://www.blueprints-ai.com")

In [70]:
# Second step: make the brochure
# Assemble all the details into another prompt to llama3.2 

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://www.blueprints-ai.com"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [73]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("RemodelAI", "https://www.blueprints-ai.com/") 

In [75]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("RemodelAI", "https://www.blueprints-ai.com")

In [ ]:
# Third Step - a minor improvement
# Change this so that the results stream back to Ollama with the familiar typewriter animation
# The model is hardcoded, so the printing in select_relevant_links will be incorrect

def stream_brochure(company_name, url,):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Blueprints", "https://www.blueprints-ai.com")

In [ ]:
# Forth Step: change for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':
# pass the model, instead of hardcoding
# correct the printing of MODEL in select_relevant_links

def stream_brochure(company_name, url, model):
    stream = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""
MODEL = "llama3.2"

stream_brochure("Blueprints", "https://www.blueprints-ai.com", "llama3.2")